# Prep

## Import stuff

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
#import seaborn as sns
#from matplotlib import pyplot as plt
#%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
#import pingouin as pg
from itertools import combinations
import openpyxl
from contextlib import redirect_stdout
import random
import math
import ast
import os

## Some magical magic to make the R stuff work

In [2]:
# The rpy2/R session and CFA machinery now live in the hitop_cfa package.
# Importing it starts the embedded R session: it sets the BLAS threading
# env vars (if unset) and loads base, utils, lavaan, and the patched semTools.
import hitop_cfa
from hitop_cfa.r_env import (ro, rbase, utils, lavaan, semtools,
                             RRuntimeError, pandas2ri)

import rpy2.ipython.html
rpy2.ipython.html.init_printing()

# Paths

In [3]:
# paths where to save preprocessed data files
log_dir = Path('./log')
log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa'
cfa_dir.mkdir(exist_ok=True)
path_save_val = fin_dir / 'dat_val.csv'
path_save_dat_gp_grid1st_norecontact = fin_dir / 'dat_gp_grid1st_norecontact.csv'
path_save_dat_en_grid1st_norecontact = fin_dir / 'dat_en_grid1st_norecontact.csv'
path_save_dat_gp_grid1st_full = fin_dir / 'dat_gp_grid1st_full.csv'
path_save_dat_en_grid1st_full = fin_dir / 'dat_en_grid1st_full.csv'
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'
# path_save_dat_gp_gridall_recontact = '../../data/finaldata/dat_gp_gridall_recontact.csv'
# path_save_dat_en_gridall_recontact = '../../data/finaldata/dat_en_gridall_recontact.csv'
# helped file for cfa
helpfile_dir = cfa_dir / 'manual_temp'
helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'
path_to_cogmood_questions = dat_dir / 'cogmood_questions.csv'
path_to_item_lookup = val_dir / 'Internalizing-Somatoform Items_DW.xlsx'


## Count how many cpus I have, then decide how many I want to use; define how many iterations for CFA (decrease for debugging)

In [4]:
import platform


def get_architecture():
    # Get the raw machine architecture string
    arch = platform.machine().lower()
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    elif "x86" in arch or "amd" in arch or "i386" in arch or "i686" in arch:
        return "x86"
    else:
        return f"Unknown ({arch})"


total_cpus = os.cpu_count()
# account for hyperthreading
arch = get_architecture()
if arch == 'ARM':
    cpus_to_use = total_cpus - 2
else:
    cpus_to_use = total_cpus // 2 - 1
print(f"\nGoing to use {cpus_to_use} CPUs for CFA heavy-lifting\n")

num_iter = 1000


Going to use 14 CPUs for CFA heavy-lifting



## SET SEEDS !!!!!!!!!!

In [5]:
#rngkind = "L'Ecuyer-CMRG"
random.seed(12345)

In [6]:
ro.r('RNGkind(kind = "L\'Ecuyer-CMRG")')
ro.r('set.seed(12345)')

<rpy2.rinterface_lib.sexp.NULLType object at 0x144824990> [0]

### TEST THE SEEDS!!!!!!!!

In [7]:
for i in range(5):
    print(random.random())
# after kernel restart, this should be 
# 0.41661987254534116
# 0.010169169457068361
# 0.8252065092537432
# 0.2986398551995928
# 0.3684116894884757

0.41661987254534116
0.010169169457068361
0.8252065092537432
0.2986398551995928
0.3684116894884757


In [8]:
ro.r('rnorm(5)')
# after kernel restart, this should be 
# -1.457850350316457	-0.45246126454182867	0.3650586371545244	-1.57091128601566	1.1419085835874878

-1.457850350316457,-0.45246126454182867,0.3650586371545244,-1.57091128601566,1.1419085835874878


### I'M ALSO SETTING THE SAME SEEDS EVERY TIME I RUN THE HELPED CFA FUNCTION, JUST IN CASE!!!!!

# Functions

## CFA helper functions

In [9]:
from hitop_cfa import (
    build_luts,
    check_hitop_ids,
    cfa_helper_func,
    run_specific_cfa,
    do_three_way_cfa_stepwise_mi,
    do_three_way_cfa_stepwise_scalar,
    exhaustive_cfa_ablations,
    exhaustive_search_scale,
    stepwise_configural_cap,
    set_seeds,
    silence_r,
)

# item-text lookups (was load_item_lookup + inline lut construction)
_luts = build_luts(path_to_item_lookup, path_to_cogmood_questions)
item_lookup = _luts['item_lookup']
item_lut = _luts['item_lut']
phq_lut = _luts['phq_lut']
gad_lut = _luts['gad_lut']
baars_lut = _luts['baars_lut']

/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/nielsond/code/hitop_val/scalar/.pixi/envs/default/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## CFA wrapper functions

# Run Main Code

## Load preprocessed data and concatinate

In [11]:
# load
data_val = pd.read_csv(path_save_val)
data_gp = pd.read_csv(path_save_dat_gp_grid1st_full)
data_en = pd.read_csv(path_save_dat_en_grid1st_full)

# concatinate
data_val_genpop = pd.concat([data_val, data_gp])
data_val_enriched = pd.concat([data_val, data_en])
data_genpop_enriched = pd.concat([data_gp, data_en])

# Dataset pairs for the hitop_cfa functions. Unified on the full
# genpop+enriched source (the old norecontact variant was column-identical
# on every CFA item, so this cannot change results).
datasets = {'val_gp': data_val_genpop, 'val_en': data_val_enriched, 'gp_en': data_genpop_enriched}
datasets_gp_en = {'gp_en': data_genpop_enriched}

# Do main CFA analysis

### 1. Find invariant subsets via exhaustive search

#### Looping through all scales where the full item set fails the scalar target:

All models go through the measEq / Wu–Estabrook ladder, and the search targets **scalar**. The search machinery (`exhaustive_search_scale`; decisions + calibration in `HANDOFF_measeq_fix.md`, PROGRESS rounds 5–6) is result-identical to testing every combination in full, but much cheaper:

- **early abandon**: pairs are tested val_gp first (most lethal) and a combination is dropped at its first failing pair/level — success still requires the full ladder on all pairs;
- **calibrated parametric pre-screen** at the delta levels (`alpha_screen = .005`; zero permutation passers lost in the 954-test calibration; configural is *never* screened — its permutation test has no parametric analog);
- **size caps** from the stepwise runs' iteration-0 configural ablations (seed-identical exhaustive refutations of all larger sizes);
- **smallest scales first**, and **per-combination progress persistence** (`cfa_dir/exhaustive_progress_{scale}.pkl`) — a killed run resumes where it stopped; completed scales short-circuit and re-print their results so the parsed log stays complete. Delete a scale's progress file to force a fresh search.

The `noninvariant_scales` list is derived from the measEq baseline (`orig_cfa_res.csv` — run `NB_2_cfa_as_reg.ipynb` first).

In [12]:
orig_items = {
    'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop77 + hitop84 + hitop92 + hitop93 + hitop123 + hitop157 + hitop182 + hitop230 + hitop246',
    'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203 + hitop240 + hitop248 + hitop265',
    'appetite_gain': 'appetite_gain =~hitop120 + hitop141 + hitop243 + hitop275',
    'appetite_loss': 'appetite_loss =~hitop280 + hitop283 + hitop109',
    'cognitive_problems': 'cognitive_problems =~hitop67 + hitop159 + hitop189 + hitop142',
    'hyposomnia': 'hyposomnia =~hitop99 + hitop181 + hitop5 + hitop66 + hitop231',
    'indecisiveness': 'indecisiveness =~hitop21 + hitop90 + hitop95',
    'insomnia': 'insomnia =~hitop160 + hitop254 + hitop261 + hitop268',
    'panic': 'panic =~hitop15 + hitop104 + hitop126 + hitop211 + hitop215 + hitop257',
    'separation_insecurity': 'separation_insecurity =~hitop40 + hitop50 + hitop69 + hitop81 + hitop113 + hitop136 + hitop151 + hitop197',
    'shame_guilt': 'shame_guilt =~hitop72 + hitop140 + hitop143 + hitop220',
    'situational_phobia': 'situational_phobia =~hitop16 + hitop165 + hitop225 + hitop247 + hitop278',
    'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop114 + hitop117 + hitop124 + hitop129 + hitop204 + hitop222 + hitop236 + hitop258',
    'well_being': 'well_being =~hitop9 + hitop23 + hitop54 + hitop106 + hitop149 + hitop200 + hitop244 + hitop245 + hitop250 + hitop281'
}

In [13]:
# Scales needing an exhaustive subset search, DERIVED from the measEq
# baseline (run the NB_2_cfa_as_reg baseline first; it writes
# orig_cfa_res.csv). The search targets SCALAR, so a scale needs it unless
# every pair's pscalar is numeric and >= .05.
orig_cfa_res = pd.read_csv(cfa_dir / 'orig_cfa_res.csv')
pscalar_num = pd.to_numeric(orig_cfa_res['pscalar'], errors='coerce')
scalar_pass_by_scale = pscalar_num.ge(0.05).groupby(orig_cfa_res['scale']).all()
noninvariant_scales = [s for s in orig_items
                       if not bool(scalar_pass_by_scale.get(s, False))]

# SEARCH ORDER: smallest scales first (cheapest information soonest)
scale_size = {s: len(orig_items[s].split('=~', 1)[1].split(' + '))
              for s in noninvariant_scales}
noninvariant_scales = sorted(noninvariant_scales, key=scale_size.get)
print(f"{len(noninvariant_scales)} scales fail 3-way scalar on the full "
      f"item set; search order (smallest first):")
print([(s, scale_size[s]) for s in noninvariant_scales])

# NOTE on the search machinery (exhaustive_search_scale; decisions and
# calibration in HANDOFF_measeq_fix.md PROGRESS rounds 5-6):
#  - pairs are tested in `datasets` dict order and a combination is
#    ABANDONED at its first failing pair/level -- keep val_gp FIRST (the
#    most-lethal pair on this data); result-identical, ~50-70% fewer
#    permutes
#  - delta levels carry the calibrated parametric pre-screen
#    (alpha_screen=.005; zero permutation passers lost in calibration;
#    configural is never screened -- different hypothesis)
#  - sizes above each scale's stepwise configural certificate are skipped
#    (stepwise_configural_cap; those sizes were exhaustively refuted by
#    seed-identical tests)
#  - progress persists per combination to
#    cfa_dir/exhaustive_progress_{scale}.pkl: a killed run RESUMES where
#    it stopped; delete a scale's progress file to force a fresh search
assert list(datasets)[0] == 'val_gp', 'val_gp must be the first pair tested'

lookingforinv_log = log_dir / 'mylog_3wayCFA_lookingforinv_seed12345_v2.txt'
with lookingforinv_log.open('w') as f:
    with redirect_stdout(f):
        for scale in noninvariant_scales:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            n = scale_size[scale]
            cap = stepwise_configural_cap(cfa_dir, scale, n)
            successes, success_size = exhaustive_search_scale(
                whichscale=scale,
                whichcfa='scalar',
                orig_items=orig_items,
                datasets=datasets,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
                min_items=3,
                max_size=cap,
                alpha_screen=0.005,
                progress_path=cfa_dir / f'exhaustive_progress_{scale}.pkl',
            )
            if successes:
                # keep this exact print pattern: the parse cells below read
                # the '!!!!!' line and take the dict from two lines later
                print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {success_size} (removing {n - success_size} items)\n')
                print(successes)

14 scales fail 3-way scalar on the full item set; search order (smallest first):
[('appetite_loss', 3), ('indecisiveness', 3), ('appetite_gain', 4), ('cognitive_problems', 4), ('insomnia', 4), ('shame_guilt', 4), ('hyposomnia', 5), ('situational_phobia', 5), ('panic', 6), ('anxious_worry', 7), ('separation_insecurity', 8), ('anhedonic_depression', 10), ('social_anxiety', 10), ('well_being', 10)]


In [14]:
lfi_out = lookingforinv_log.read_text().split('\n')



In [15]:
lfi_parsing = []
for lix, line in enumerate(lfi_out):
    if line.startswith('!!!!!'):
        row= dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        lfi_parsing.append(row)


In [16]:
ds_pairs = ['val_gp', 'val_en', 'gp_en']
# measEq / Wu-Estabrook ladder: cfa_helper_func now reports a thresholds
# level between config and metric
inv_levels = ['config', 'thresholds', 'metric', 'scalar', 'strict']

In [17]:
inv_dat = []
for lprow in lfi_parsing:
    scale_stats = ast.literal_eval(lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        ogformula = orig_items[lprow['scale']]
        itemformula_only = ogformula.split("=~",1)[1]
        ogitems = itemformula_only.split(" + ")
        row = lprow.copy()
        item_nos = iss[0]
        items = [item_lut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [item_lut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        inv_dat.append(row)
inv_dat = pd.DataFrame(inv_dat)
if inv_dat.empty:
    # no exhaustive successes at all: keep the schema so the review-note
    # merge and pickle cells below still run instead of crashing
    inv_dat = pd.DataFrame(columns=['lix', 'scale', 'n_items', 'n_removed',
                                    'ssix', 'item_nos', 'removed_nos',
                                    'items', 'removed'])

In [18]:
inv_dat

,lix,scale,n_items,n_removed,ssix,item_nos,removed_nos,items,removed,val_gp__config,val_gp__thresholds,val_gp__metric,val_gp__scalar,val_gp__strict,val_en__config,val_en__thresholds,val_en__metric,val_en__scalar,val_en__strict,gp_en__config,gp_en__thresholds,gp_en__metric,gp_en__scalar,gp_en__strict
0,46,insomnia,3,1,0,"(hitop160, hitop254, hitop268)",[hitop261],"[I had trouble staying asleep., I slept very poorly., I lay awake for a long time before falling...",[I woke up early and could not get back to sleep.],0.797,0.219,0.581,0.548,NaN,0.875,0.615,0.910,0.361,NaN,0.651,0.291,0.558,0.084,NaN
1,77,situational_phobia,3,2,0,"(hitop16, hitop165, hitop247)","[hitop225, hitop278]","[I avoided riding in elevators., I was afraid of flying., I was afraid of heights.]","[I was afraid of the dark., I became very anxious during a storm.]",0.103,0.822,0.076,0.096,NaN,0.249,0.922,0.370,0.534,NaN,0.519,0.996,0.521,0.673,NaN
2,90,panic,3,3,0,"(hitop15, hitop104, hitop215)","[hitop126, hitop211, hitop257]","[I was short of breath., I felt nauseated., My hands were cold or sweaty.]","[My heart was racing or pounding., I was trembling or shaking., I felt dizzy or lightheaded.]",0.326,0.144,0.672,0.140,NaN,0.872,0.515,0.995,0.530,NaN,0.442,0.409,0.609,0.581,NaN
3,90,panic,3,3,1,"(hitop104, hitop126, hitop215)","[hitop15, hitop211, hitop257]","[I felt nauseated., My heart was racing or pounding., My hands were cold or sweaty.]","[I was short of breath., I was trembling or shaking., I felt dizzy or lightheaded.]",0.330,0.825,0.513,0.627,NaN,0.930,0.804,0.604,0.943,NaN,0.507,0.560,0.931,0.628,NaN
4,90,panic,3,3,2,"(hitop104, hitop211, hitop215)","[hitop15, hitop126, hitop257]","[I felt nauseated., I was trembling or shaking., My hands were cold or sweaty.]","[I was short of breath., My heart was racing or pounding., I felt dizzy or lightheaded.]",0.599,0.824,0.304,0.334,NaN,0.825,0.630,0.699,0.267,NaN,0.040,0.412,0.743,0.864,NaN
5,90,panic,3,3,3,"(hitop126, hitop211, hitop215)","[hitop15, hitop104, hitop257]","[My heart was racing or pounding., I was trembling or shaking., My hands were cold or sweaty.]","[I was short of breath., I felt nauseated., I felt dizzy or lightheaded.]",0.980,0.928,0.087,0.205,NaN,0.789,0.784,0.098,0.177,NaN,0.894,0.787,0.958,0.459,NaN
6,112,separation_insecurity,7,1,0,"(hitop40, hitop50, hitop69, hitop81, hitop136, hitop151, hitop197)",[hitop113],"[I felt insecure about important relationships in my life., I wanted someone else to make decisi...",[I could not handle rejection.],0.072,0.474,0.150,0.188,NaN,0.111,0.898,0.136,0.104,NaN,0.108,0.985,0.302,0.262,NaN
7,134,social_anxiety,4,6,0,"(hitop114, hitop117, hitop129, hitop204)","[hitop1, hitop17, hitop124, hitop222, hitop236, hitop258]","[I had difficulty making eye contact with others., I felt socially awkward., I avoided performin...","[I felt shy around other people., I was uncomfortable meeting new people., I avoided situations ...",0.588,0.096,0.922,0.127,NaN,0.926,0.796,0.648,0.135,NaN,0.546,0.407,0.738,0.059,NaN
8,147,well_being,3,7,0,"(hitop54, hitop244, hitop250)","[hitop9, hitop23, hitop106, hitop149, hitop200, hitop245, hitop281]","[It was easy for me to laugh., I felt optimistic., I felt like I had a lot of interesting things...","[I felt like I was having a lot of fun., I felt cheerful., I was proud of myself., I felt good a...",0.498,0.561,0.374,0.281,NaN,0.755,0.430,0.277,0.938,NaN,0.330,0.059,0.848,0.351,NaN
9,147,well_being,3,7,1,"(hitop106, hitop244, hitop250)","[hitop9, hitop23, hitop54, hitop149, hitop200, hitop245, hitop281]","[I was proud of myself., I felt optimistic., I felt like I had a lot of interesting things to do.]","[I felt like I was having a lot of fun., I felt cheerful., It was easy for me to laugh., I felt ...",0.774,0.514,0.464,0.163,NaN,0.742,0.482,0.069,0.056,NaN,0.367,0.083,0.378,0.224,NaN


In [22]:
for row in inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for kk, ii in zip(row.removed_nos, row.removed):
        print(kk, ':', ii)
    print()
    print()

########################
Scale: insomnia, Subset_id: 0
########################
I had trouble staying asleep.
I slept very poorly.
I lay awake for a long time before falling asleep.
---------REMOVED---------------
hitop261 : I woke up early and could not get back to sleep.


########################
Scale: situational_phobia, Subset_id: 0
########################
I avoided riding in elevators.
I was afraid of flying.
I was afraid of heights.
---------REMOVED---------------
hitop225 : I was afraid of the dark.
hitop278 : I became very anxious during a storm.


########################
Scale: panic, Subset_id: 0
########################
I was short of breath.
I felt nauseated.
My hands were cold or sweaty.
---------REMOVED---------------
hitop126 : My heart was racing or pounding.
hitop211 : I was trembling or shaking.
hitop257 : I felt dizzy or lightheaded.


########################
Scale: panic, Subset_id: 1
########################
I felt nauseated.
My heart was racing or pounding.
M

In [24]:
pd.read_pickle(cfa_dir/ 'stepwise_scalar.pkl')

,scale,core_level,item_nos,removed_nos,items,removed,metric_item_nos,scalar_removed_nos,error
0,anhedonic_depression,metric,"[hitop77, hitop84, hitop92, hitop123, hitop182, hitop230]","[hitop39, hitop93, hitop157, hitop246]","[I didn’t look forward to seeing friends or family., I felt depressed., It took a lot of effort ...","[It felt like there wasn’t anything interesting or fun to do., Nothing seemed interesting to me....","[hitop77, hitop84, hitop92, hitop123, hitop182, hitop230]","[hitop123, hitop84, hitop182, hitop77]",NaN
1,anxious_worry,metric,"[hitop240, hitop248, hitop265]","[hitop20, hitop34, hitop89, hitop203]","[I felt nervous and ""on edge""., I worried about almost everything., I was overwhelmed by anxiety.]","[I felt tense., Thoughts were racing through my head., I had a lot of nervous energy., I felt ve...","[hitop240, hitop248, hitop265]",[hitop248],NaN
2,appetite_gain,metric,"[hitop141, hitop243, hitop275]",[hitop120],"[I thought a lot about food., I stuffed myself with food., I ate even when I was not really hung...",[I could not keep myself from eating.],"[hitop141, hitop243, hitop275]",[hitop141],NaN
3,appetite_loss,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cognitive_problems,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,hyposomnia,metric,"[hitop99, hitop5, hitop66, hitop231]",[hitop181],"[I needed much less sleep than usual., I had days when I never got tired., I did not feel tired,...",[I felt like I could keep going and going without ever getting tired.],"[hitop99, hitop5, hitop66, hitop231]","[hitop5, hitop231]",NaN
6,indecisiveness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,insomnia,scalar,"[hitop160, hitop254, hitop268]",[hitop261],"[I had trouble staying asleep., I slept very poorly., I lay awake for a long time before falling...",[I woke up early and could not get back to sleep.],"[hitop160, hitop254, hitop261, hitop268]",[hitop261],NaN
8,panic,scalar,"[hitop104, hitop126, hitop215]","[hitop15, hitop211, hitop257]","[I felt nauseated., My heart was racing or pounding., My hands were cold or sweaty.]","[I was short of breath., I was trembling or shaking., I felt dizzy or lightheaded.]","[hitop15, hitop104, hitop126, hitop215, hitop257]","[hitop257, hitop15]",NaN
9,separation_insecurity,scalar,"[hitop40, hitop69, hitop81, hitop136, hitop151, hitop197]","[hitop50, hitop113]","[I felt insecure about important relationships in my life., I wanted other people to take care o...","[I wanted someone else to make decisions for me., I could not handle rejection.]","[hitop40, hitop69, hitop81, hitop113, hitop136, hitop151, hitop197]",[hitop113],NaN


In [27]:
review_notes = [
    {
        'scale': 'insomnia',
        'ssix': 0,
        'note': 'Single solution',
        'stepwise_agrees':True
    },
    {
        'scale': 'situational_phobia',
        'ssix': 0,
        'note': 'Single solution',
        'stepwise_agrees': False
    },
    {
        'scale': 'panic',
        'ssix': 1,
        'note': "scale is just the the physical symptoms of panic attack, arbitrary to pick one, but this agrees with stepwise_scalar",
        'stepwise_agrees': True
    },
    {
        'scale': 'separation_insecurity', 
        'ssix': 0,
        'note': "Single solution, could not handle rejection is an ambiguous item, what does failing to handle mean here",
        'stepwise_agrees': False
    },
    {
        'scale': 'social_anxiety', 
        'ssix': 0,
        'note': "Single solution at stopping size",
        'stepwise_agrees': False
    },
    {
        'scale': 'well_being',
        'ssix': 1,
        'note': "I think being proud of yourself is a better indicator of well-being than laughing easily, but it's arbitrary",
        'stepwise_agrees': False
    }
]
review_notes = pd.DataFrame(review_notes)
review_notes['exhaustive_choice'] = True


In [28]:
review_notes

,scale,ssix,note,stepwise_agrees,exhaustive_choice
0,insomnia,0,Single solution,True,True
1,situational_phobia,0,Single solution,False,True
2,panic,1,"scale is just the the physical symptoms of panic attack, arbitrary to pick one, but this agrees ...",True,True
3,separation_insecurity,0,"Single solution, could not handle rejection is an ambiguous item, what does failing to handle me...",False,True
4,social_anxiety,0,Single solution,False,True
5,well_being,1,"I think being proud of yourself is a better indicator of well-being than laughing easily, but it...",False,True


In [29]:
inv_dat = inv_dat.merge(review_notes, how='outer', on=['scale', 'ssix'])
inv_dat['exhaustive_choice'] = inv_dat.exhaustive_choice.fillna(False)

In [30]:
inv_dat.to_pickle(cfa_dir / 'exhaustive_scalar.pkl')

In [31]:
inv_dat.loc[inv_dat.exhaustive_choice]

,lix,scale,n_items,n_removed,ssix,item_nos,removed_nos,items,removed,val_gp__config,val_gp__thresholds,val_gp__metric,val_gp__scalar,val_gp__strict,val_en__config,val_en__thresholds,val_en__metric,val_en__scalar,val_en__strict,gp_en__config,gp_en__thresholds,gp_en__metric,gp_en__scalar,gp_en__strict,note,stepwise_agrees,exhaustive_choice
0,46,insomnia,3,1,0,"(hitop160, hitop254, hitop268)",[hitop261],"[I had trouble staying asleep., I slept very poorly., I lay awake for a long time before falling...",[I woke up early and could not get back to sleep.],0.797,0.219,0.581,0.548,NaN,0.875,0.615,0.910,0.361,NaN,0.651,0.291,0.558,0.084,NaN,Single solution,True,True
2,90,panic,3,3,1,"(hitop104, hitop126, hitop215)","[hitop15, hitop211, hitop257]","[I felt nauseated., My heart was racing or pounding., My hands were cold or sweaty.]","[I was short of breath., I was trembling or shaking., I felt dizzy or lightheaded.]",0.330,0.825,0.513,0.627,NaN,0.930,0.804,0.604,0.943,NaN,0.507,0.560,0.931,0.628,NaN,"scale is just the the physical symptoms of panic attack, arbitrary to pick one, but this agrees ...",True,True
5,112,separation_insecurity,7,1,0,"(hitop40, hitop50, hitop69, hitop81, hitop136, hitop151, hitop197)",[hitop113],"[I felt insecure about important relationships in my life., I wanted someone else to make decisi...",[I could not handle rejection.],0.072,0.474,0.150,0.188,NaN,0.111,0.898,0.136,0.104,NaN,0.108,0.985,0.302,0.262,NaN,"Single solution, could not handle rejection is an ambiguous item, what does failing to handle me...",False,True
6,77,situational_phobia,3,2,0,"(hitop16, hitop165, hitop247)","[hitop225, hitop278]","[I avoided riding in elevators., I was afraid of flying., I was afraid of heights.]","[I was afraid of the dark., I became very anxious during a storm.]",0.103,0.822,0.076,0.096,NaN,0.249,0.922,0.370,0.534,NaN,0.519,0.996,0.521,0.673,NaN,Single solution,False,True
7,134,social_anxiety,4,6,0,"(hitop114, hitop117, hitop129, hitop204)","[hitop1, hitop17, hitop124, hitop222, hitop236, hitop258]","[I had difficulty making eye contact with others., I felt socially awkward., I avoided performin...","[I felt shy around other people., I was uncomfortable meeting new people., I avoided situations ...",0.588,0.096,0.922,0.127,NaN,0.926,0.796,0.648,0.135,NaN,0.546,0.407,0.738,0.059,NaN,Single solution,False,True
9,147,well_being,3,7,1,"(hitop106, hitop244, hitop250)","[hitop9, hitop23, hitop54, hitop149, hitop200, hitop245, hitop281]","[I was proud of myself., I felt optimistic., I felt like I had a lot of interesting things to do.]","[I felt like I was having a lot of fun., I felt cheerful., It was easy for me to laugh., I felt ...",0.774,0.514,0.464,0.163,NaN,0.742,0.482,0.069,0.056,NaN,0.367,0.083,0.378,0.224,NaN,"I think being proud of yourself is a better indicator of well-being than laughing easily, but it...",False,True


### 3. (if needed) Check invariance of any excluded subsets that have 3 or more items

The subsets are **derived** from the final-cores table (`stepwise_scalar.pkl`, written by the scalar-continuation loop in `NB_2_cfa_as_reg.ipynb`): for every scale whose final core dropped 3+ items, the removed items are tested as their own scale up the full ladder.

In [32]:
# Excluded subsets with >= 3 items, derived from the final cores
# (previously a hardcoded dict of outcome-specific formulas)
final_cores = pd.read_pickle(cfa_dir / 'stepwise_scalar.pkl')
to_test_inv = {}
for row in final_cores.itertuples():
    removed = getattr(row, 'removed_nos', None)
    if isinstance(removed, list) and len(removed) >= 3:
        to_test_inv[row.scale] = f"{row.scale} =~" + " + ".join(removed)
print(f"{len(to_test_inv)} excluded subset(s) with >= 3 items:")
print(to_test_inv)

with open(log_dir / "mylog_3wayCFA_invsubsets_seed12345_2.txt", "w") as f:
    with redirect_stdout(f):
        for scale, items in to_test_inv.items():
            # create a neat list of items to test for this scale
            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")
            # test
            run_specific_cfa(
                whichscale=scale,
                item_list=items_list,
                whichcfa='strict',
                datasets=datasets,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
            )

5 excluded subset(s) with >= 3 items:
{'anhedonic_depression': 'anhedonic_depression =~hitop39 + hitop93 + hitop157 + hitop246', 'anxious_worry': 'anxious_worry =~hitop20 + hitop34 + hitop89 + hitop203', 'panic': 'panic =~hitop15 + hitop211 + hitop257', 'social_anxiety': 'social_anxiety =~hitop1 + hitop17 + hitop117 + hitop258', 'well_being': 'well_being =~hitop9 + hitop54 + hitop106 + hitop200 + hitop244 + hitop245 + hitop281'}


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

## For interpretation - check what items mean

In [33]:
# Item-id lists per scale, generated from orig_items (the old hardcoded
# dict had drifted: it was missing indecisiveness entirely and dropped
# some items, e.g. 240 from anxious_worry and 159 from cognitive_problems)
scales = {
    f'items_{scale}': [ii.strip().replace('hitop', '')
                       for ii in formula.split('=~', 1)[1].split(' + ')]
    for scale, formula in orig_items.items()
}

for scale, items in scales.items():
    print(scale)
    check_hitop_ids(items, my_item_lookup=item_lookup)

items_anhedonic_depression
   hitop39   It felt like there wasn’t anything interesting or fun to do.
   hitop77   I didn’t look forward to seeing friends or family.
   hitop84   I felt depressed.
   hitop92   It took a lot of effort to do everyday activities.
   hitop93   Nothing seemed interesting to me.
   hitop123   Nothing made me laugh.
   hitop157   I had very little energy.
   hitop182   I was unable to enjoy things like I normally do.
   hitop230   I felt emotionally numb.
   hitop246   I was a lot less talkative than usual.


items_anxious_worry
   hitop20   I felt tense.
   hitop34   Thoughts were racing through my head.
   hitop89   I had a lot of nervous energy.
   hitop203   I felt very stressed.
   hitop240   I felt nervous and "on edge".
   hitop248   I worried about almost everything.
   hitop265   I was overwhelmed by anxiety.


items_appetite_gain
   hitop120   I could not keep myself from eating.
   hitop141   I thought a lot about food.
   hitop243   I stuffed mysel

# Analysis for BAARS, GAD, PHQ

In [34]:
other_scales = {
    'phq_sum': 'phq_sum=~phq_1 + phq_2 + phq_3 + phq_4 + phq_5 + phq_6 + phq_7 + phq_8',
    'gad_sum': 'gad_sum =~gad_1 + gad_2 + gad_3 + gad_4 + gad_5 + gad_6 + gad_7',
    'baars_inattention_sum': 'baars_inattention_sum =~inattention_1 + inattention_2 + inattention_3 + inattention_4 + inattention_5 + inattention_6 + inattention_7 + inattention_8 + inattention_9',
    'baars_hyperactivity_sum': 'baars_hyperactivity_sum =~hyperactivity_1 + hyperactivity_2 + hyperactivity_3 + hyperactivity_4 + hyperactivity_5',
    'baars_impulsivity_sum': 'baars_impulsivity_sum =~impulsivity_1 + impulsivity_2 + impulsivity_3 + impulsivity_4',
    'baars_sct_sum': 'baars_sct_sum =~sct_1 + sct_2 + sct_3 + sct_4 + sct_5 + sct_6 + sct_7 + sct_8 + sct_9'}

# item-text lookup per other scale (keyed by scale name so downstream cells
# no longer depend on a hardcoded, order-aligned list)
other_luts = {
    'phq_sum': phq_lut,
    'gad_sum': gad_lut,
    'baars_inattention_sum': baars_lut['inattention'],
    'baars_hyperactivity_sum': baars_lut['hyperactivity'],
    'baars_impulsivity_sum': baars_lut['impulsivity'],
    'baars_sct_sum': baars_lut['sct'],
}

other_log = log_dir /'mylog_CFA_baarsgadphq_seed12345.txt'
with other_log.open("w") as f:
    with redirect_stdout(f):
        other_cfa_res = []
        for scale, items in other_scales.items():
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            print(f"Items: {items}")

            items_only = items.split("=~",1)[1]
            items_list = items_only.split(" + ")

            # +++ GENPOP VS ENRICHED +++
            print('\n -----> GENPOP VS ENRICHED <----- ')
            (flag_metric, pconfig, pthresholds, pmetric,
             pscalar, pstrict) = cfa_helper_func(
                            scalename=scale,
                            list_of_items=items_list,
                            mydata_python=data_genpop_enriched,
                            mydata_temp_path=path_to_helpfile,
                            num_iter=num_iter,
                            cpus_to_use=cpus_to_use,
                            max_level='strict')
            other_cfa_res.append(dict(
                pair='gp_en', scale=scale, pconfig=pconfig,
                pthresholds=pthresholds, pmetric=pmetric,
                pscalar=pscalar, pstrict=pstrict))
other_cfa_res = pd.DataFrame(other_cfa_res)
other_cfa_res.to_csv(cfa_dir / 'other_orig_cfa_res.csv', index=None)
other_cfa_res

R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be perm

,pair,scale,pconfig,pthresholds,pmetric,pscalar,pstrict
0,gp_en,phq_sum,0.000,NA,NA,NA,NA
1,gp_en,gad_sum,0.065,0.28,0.301,0.244,0.056
2,gp_en,baars_inattention_sum,0.000,NA,NA,NA,NA
3,gp_en,baars_hyperactivity_sum,0.147,0.676,0.415,0.229,0.084
4,gp_en,baars_impulsivity_sum,0.481,0.376,0.352,0.31,0.704
5,gp_en,baars_sct_sum,0.000,NA,NA,NA,NA


In [ ]:
# Other scales needing an exhaustive subset search, DERIVED from the
# other-scales baseline above: a scale needs it unless its gp_en pscalar
# is numeric and >= .05 (these searches are single-pair, gp_en only).
other_pscalar = pd.to_numeric(other_cfa_res['pscalar'], errors='coerce')
other_scalar_pass = other_pscalar.ge(0.05).groupby(other_cfa_res['scale']).all()
other_failing_scales = [s for s in other_scales
                        if not bool(other_scalar_pass.get(s, False))]
# smallest scales first
other_size = {s: len(other_scales[s].split('=~', 1)[1].split(' + '))
              for s in other_failing_scales}
other_failing_scales = sorted(other_failing_scales, key=other_size.get)
print(f"{len(other_failing_scales)} of {len(other_scales)} other scales "
      f"fail scalar on the full item set; search order (smallest first):")
print([(s, other_size[s]) for s in other_failing_scales])

# same machinery as the HiTOP search (single pair, so early abandon is
# within-ladder only; no stepwise runs exist for these scales -> no caps)
other_exhaustive_logs = log_dir /'mylog_exhuastive_inv_search_baarsgadphq_seed12345.txt'

with other_exhaustive_logs.open("w") as f:
    with redirect_stdout(f):
        for scale in other_failing_scales:
            print(f'\n\n\n======================================\nTESTING SCALE {scale.upper()}\n======================================\n')
            n = other_size[scale]
            successes, success_size = exhaustive_search_scale(
                whichscale=scale,
                whichcfa='scalar',
                orig_items=other_scales,
                datasets=datasets_gp_en,
                temp_path=path_to_helpfile,
                num_iter=num_iter,
                cpus_to_use=cpus_to_use,
                min_items=3,
                max_size=None,
                alpha_screen=0.005,
                progress_path=cfa_dir / f'exhaustive_progress_{scale}.pkl',
            )
            if successes:
                # keep this exact print pattern: the parse cells below read
                # the '!!!!!' line and take the dict from two lines later
                print(f'\n!!!!! Found at least one invariant subset for SCALE {scale} with ITEMS = {success_size} (removing {n - success_size} items)\n')
                print(successes)

3 of 6 other scales fail scalar on the full item set; search order (smallest first):
[('phq_sum', 8), ('baars_inattention_sum', 9), ('baars_sct_sum', 9)]


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.


R[write to console]: No AFIs were selected, so only chi-squared will be permuted.




In [40]:
other_lfi_out = other_exhaustive_logs.read_text().split('\n')
other_lfi_parsing = []
for lix, line in enumerate(other_lfi_out):
    if line.startswith('!!!!!'):
        row = dict(
            lix=lix,
            scale=line.split('SCALE')[-1].split('with')[0].strip(),
            n_items=int(line.split('ITEMS = ')[-1].split(' (')[0]),
            n_removed=int(line.split('removing ')[-1].split(' items')[0])
        )
        other_lfi_parsing.append(row)
other_ds_pairs = ['gp_en']

In [41]:
other_inv_dat = []
for lprow in other_lfi_parsing:
    # item-text lookup resolved by scale name (previously a hardcoded list
    # of luts that had to stay order-aligned with a hardcoded scale list)
    olut = other_luts[lprow['scale']]
    scale_stats = ast.literal_eval(other_lfi_out[lprow['lix'] + 2])
    for issix, iss in enumerate(scale_stats.items()):
        row = lprow.copy()
        item_nos = iss[0]
        ogitems = list(olut.keys())
        items = [olut[item_no] for item_no in item_nos]
        removed_nos = [ii for ii in ogitems if ii not in item_nos]
        removed_items = [olut[item_no] for item_no in removed_nos]
        row['ssix'] = issix
        row['item_nos'] = item_nos
        row['removed_nos'] = removed_nos
        row['items'] = items
        row['removed'] = removed_items
        inv_stats = iss[1]
        for dsp in other_ds_pairs:
            for lix, level in enumerate(inv_levels):
                p = inv_stats[dsp][lix]
                if p == 'NA':
                    p = np.nan
                else:
                    p = float(p)
                row[f'{dsp}__{level}'] = p
        other_inv_dat.append(row)
other_inv_dat = pd.DataFrame(other_inv_dat)
if other_inv_dat.empty:
    # no exhaustive successes: keep the schema so downstream cells run
    other_inv_dat = pd.DataFrame(columns=['lix', 'scale', 'n_items',
                                          'n_removed', 'ssix', 'item_nos',
                                          'removed_nos', 'items', 'removed'])

In [42]:
for row in other_inv_dat.itertuples():
    print("########################")
    print(f'Scale: {row.scale}, Subset_id: {row.ssix}')
    print("########################")
    for ii in row.items:
        print(ii)
    print('---------REMOVED---------------')
    for kk, ii in zip(row.removed_nos, row.removed):
        print(kk, ':', ii)
    print()
    print()

########################
Scale: phq_sum, Subset_id: 0
########################
Little interest or pleasure in doing things
Feeling down, depressed, irritable or hopeless
Feeling tired or having little energy
Feeling bad about yourself – or that you are a failure or have let yourself or your family down
Moving or speaking so slowly that other people could have noticed? Or the opposite – being so fidgety or restless that you have been moving around a lot more than usual
---------REMOVED---------------
phq_3 : Trouble falling or staying asleep, or sleeping too much
phq_5 : Poor appetite or overeating
phq_7 : Trouble concentrating on things, such as school work, reading or watching television


########################
Scale: baars_inattention_sum, Subset_id: 0
########################
Fail to give close attention to details or make careless mistakes in my work or other activities
Difficulty sustaining my attention in tasks or fun activities
Don't listen when spoken to directly
Have diffic

In [43]:
other_review_notes = [
    {
        'scale': 'phq_sum',
        'ssix': 0,
        'note': 'Only solution with 3 items removed, may be items with different interpretations in clinical and non-clincial populations',
    },
    {
        'scale': 'baars_inattention_sum',
        'ssix': 3,
        'note': 'Not listening when spoken to directly and being forgetful in everyday activities seem more central to this to me so I picked a solution that retained them',
    },
    {
        'scale': 'baars_sct_sum',
        'ssix': 1,
        'note': 'Slow moving could have other causes',
    },
]
other_review_notes = pd.DataFrame(other_review_notes)
other_review_notes['exhaustive_choice'] = True


In [44]:
other_inv_dat = other_inv_dat.merge(other_review_notes, how='outer', on=['scale', 'ssix'])
other_inv_dat['exhaustive_choice'] = other_inv_dat.exhaustive_choice.fillna(False)


In [45]:
other_inv_dat.to_pickle(cfa_dir / 'other_scales_exhaustive.pkl')